# NoteMap: Visualizing Domain Knowledge Graphically

# Ingestion

In [3]:
from pathlib import Path
from IPython.display import Image, display

import os
from anthropic import Anthropic
from openai import OpenAI
from dotenv import load_dotenv


from notemap.ingest.pdf import rasterize_pdf, transcribe_page, save_pdfs, ingest_pdf
from notemap.cache import content_hash, cache_path
from notemap.embeddings.chunking import chunks_from_pdf
from notemap.embeddings.embed import chunks_to_embeddings, load_pdf_embeddings
from notemap.clustering.umap import umap_reduce, plot_embeddings, hdb_cluster, plot_embedding_clusters
from notemap.clustering.nodes import create_layout, create_centroid_nodes, compute_edges

%load_ext autoreload
%autoreload 2

/opt/anaconda3/envs/NoteMap/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
PDF_IN_PATH = Path("notes/pdfs")
DATA_PATH = Path("data")
manifest_path = DATA_PATH / "embeddings" / "manifest.json"

BATCH_SIZE = 256
SEED = 777
load_dotenv()

ocr_client = Anthropic()
embedding_client = OpenAI()
ocr_model = "claude-haiku-4-5"
embedding_model = "text-embedding-3-small"



In [48]:
save_pdfs(PDF_IN_PATH, DATA_PATH, client=ocr_client, model=ocr_model)

Ingested and saved JSON data for  data/documents/Lecture 2- Optim_945206f5.json
Ingested and saved JSON data for  data/documents/Geometric Optics_e8d0eb63.json
Ingested and saved JSON data for  data/documents/Principle Ray Diagrams_ef6ce560.json
Ingested and saved JSON data for  data/documents/duality continued_7bad7fa3.json
Ingested and saved JSON data for  data/documents/EE120-FA24-DISC02_2d9aa91e.json
Ingested and saved JSON data for  data/documents/pitchbook_prep_77a66636.json
Ingested and saved JSON data for  data/documents/Lecture 7- GDA_0d742f77.json
Ingested and saved JSON data for  data/documents/CNNS_cde2d033.json
Ingested and saved JSON data for  data/documents/Lec8- IAGS, CT State Space_b262eb9b.json
Ingested and saved JSON data for  data/documents/Lecture 13- Convexity_f3001efc.json
Ingested and saved JSON data for  data/documents/HW-10_f9ec0abb.json
Ingested and saved JSON data for  data/documents/AVL Trees_4bff18ad.json
Ingested and saved JSON data for  data/documents/PE

In [15]:
embeddings, chunk_ids = load_pdf_embeddings(DATA_PATH, client=embedding_client, model=embedding_model, batch_size=BATCH_SIZE)
print(embeddings.shape)

(677, 1536)


## UMAP (DIM REDUCTION) and Clustering

#### Reduce from ~1536 dimensions to ~40 for HDBScan -> 2 dimensions for visualizing clusters

In [16]:
UMAP1_PARAMS = {
    "n_neighbors": 15,
    "n_components": 40,
    "min_distance": 0.0
}

HDB_PARAMS = {
    "min_samples": 1,
    "min_cluster_size": 7,
    "method": "eom"
}

UMAP2_PARAMS = {
    "n_neighbors": 20,
    "n_components": 2,
    "min_distance": 0.6
}

In [17]:
create_layout(embeddings, HDB_PARAMS, UMAP1_PARAMS, UMAP2_PARAMS, client=ocr_client, model=ocr_model, SEED=SEED)

Unlabeled points: 109/677
(677, 2)
(677,)
Number of Clusters:  27
Generating Cluster Labels... 
Cluster Labels Generated


In [ ]:
compute_edges(embeddings)

Edge (0, 1) with weight 0.2315983742001808
Edge (0, 485) with weight 0.33349675257965183
Edge (1, 151) with weight 0.201636236426031
Edge (1, 470) with weight 0.23859622258184354
Edge (1, 646) with weight 0.3116416928513156
Edge (2, 74) with weight 0.23255089901634596
Edge (3, 413) with weight 0.2516145927426443
Edge (3, 631) with weight 0.2841866075518821
Edge (4, 580) with weight 0.3093144752780921
Edge (5, 6) with weight 0.3062975919493003
Edge (5, 53) with weight 0.2831791811625566
Edge (5, 478) with weight 0.24927814665810433
Edge (6, 479) with weight 0.3812940956794545
Edge (7, 44) with weight 0.2745971451974578
Edge (7, 492) with weight 0.3704890266659945
Edge (8, 165) with weight 0.39688976055027103
Edge (9, 10) with weight 0.37093856749267884
Edge (10, 273) with weight 0.3794383233516868
Edge (11, 248) with weight 0.4226180464697168
Edge (12, 19) with weight 0.29174772869914756
Edge (12, 249) with weight 0.3201102524107027
Edge (13, 247) with weight 0.27951711871910323
Edge (1